[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E6%95%B8%E6%93%9A%E6%95%B4%E7%90%86_%E9%80%A3%E6%8E%A5_%E5%90%88%E4%BD%B5_%E9%87%8D%E5%A1%91/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%8C%B6%E8%AA%9E%E6%99%82%E5%85%89_%E4%B8%89%E5%80%8B%E6%9C%88%E8%A8%82%E5%96%AE%E5%90%88%E4%BD%B5%E6%88%90%E5%AD%A3%E5%A0%B1.ipynb)

# 🔗 茶語時光：把三個月的訂單合併成第一季季報

> 🧋 **情境**：POS 系統每個月匯出一個訂單檔，而且為了省空間，檔案裡只有「門市代碼」和「品項代碼」，沒有名稱、也沒有價格。老闆要一份第一季季報：「每家店每個月賣了多少錢？哪一類飲料成長最多？」你需要把 3 個月份的檔案**接起來**，再和品項主檔、門市主檔**配對**，最後**轉成老闆看得懂的報表版面**。

| 素材檔案 | 內容 |
| --- | --- |
| `訂單_2024-01.csv ～ 訂單_2024-03.csv` | 第一季三個月的訂單明細（只有代碼，沒有名稱和價格） |
| `品項主檔.csv` | 品項代碼 → 品項名稱、類別、價格、成本 |
| `門市主檔.csv` | 門市代碼 → 門市名稱、縣市 |

> ⚠️ 「茶語時光」是課程虛構的品牌，資料為模擬資料；節日與颱風假的日期參考台灣實際情況，方便你對照生活經驗。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '訂單_2024-01.csv',
    '訂單_2024-02.csv',
    '訂單_2024-03.csv',
    '品項主檔.csv',
    '門市主檔.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E6%95%B8%E6%93%9A%E6%95%B4%E7%90%86_%E9%80%A3%E6%8E%A5_%E5%90%88%E4%BD%B5_%E9%87%8D%E5%A1%91/%E6%95%B8%E6%93%9A%E6%95%B4%E7%90%86_%E9%80%A3%E6%8E%A5_%E5%90%88%E4%BD%B5_%E9%87%8D%E5%A1%91_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['訂單_2024-01.csv', '訂單_2024-02.csv', '訂單_2024-03.csv', '品項主檔.csv', '門市主檔.csv']


## 任務 1：concat——把三個月的訂單「上下接起來」

**先踩一個陷阱**：直接讀檔，看看「品項代碼」變成什麼？

In [2]:
import pandas as pd

trap = pd.read_csv('訂單_2024-01.csv')
print('沒指定型態時的品項代碼：', trap['品項代碼'].unique()[:5], trap['品項代碼'].dtype)   # 001 變成 1 了！

沒指定型態時的品項代碼： [4 5 6 3 1] int64


In [3]:
months = []
for m in ['01', '02', '03']:
    df = pd.read_csv(f'訂單_2024-{m}.csv', dtype={'品項代碼': str})   # 代碼一律當文字讀，保住開頭的 0
    df['月份'] = f'{int(m)}月'                                         # 加上來源月份
    months.append(df)

orders = pd.concat(months, ignore_index=True)                          # 上下接起來，重新編列號
print('各月筆數：', [len(m) for m in months], '→ 合計', len(orders))
print(orders.head())

各月筆數： [38576, 36667, 45684] → 合計 120927
            訂單編號                 訂單時間 門市代碼 品項代碼 尺寸  數量  月份
0  T240101059972  2024-01-01 10:03:02  S05  004  中   1  1月
1  T240101059911  2024-01-01 10:06:22  S05  005  中   1  1月
2  T240101059907  2024-01-01 10:10:32  S05  006  中   1  1月
3  T240101059907  2024-01-01 10:10:32  S05  005  大   1  1月
4  T240101059907  2024-01-01 10:10:32  S05  003  大   1  1月


> 💡 **看懂結果**：沒指定型態時，`001` 被當成數字 `1`，等一下和主檔的 `'001'` 就**配不起來**了。代碼、電話、郵遞區號這類「長得像數字的文字」，讀檔時一定要指定成文字。

## 任務 2：merge——用品項代碼查出名稱與價格（就是 VLOOKUP）

In [4]:
items = pd.read_csv('品項主檔.csv', dtype={'品項代碼': str})
print(items)

full = orders.merge(items, on='品項代碼', how='left', validate='many_to_one')   # 多筆訂單對一個品項；validate 幫你檢查主檔沒有重複
full['單價'] = full['中杯價'] + (full['尺寸'] == '大') * 10                   # 大杯 +10 元
full['金額'] = full['單價'] * full['數量']
full['毛利'] = (full['單價'] - full['成本']) * full['數量']
print('\n合併前後筆數：', len(orders), '→', len(full), '（how=left 時兩者應該相同）')
print(full[['訂單編號', '品項代碼', '品項', '尺寸', '數量', '單價', '金額']].head())

  品項代碼      品項  類別  中杯價  成本
0  001    珍珠奶茶  奶茶   55  17
1  002   四季春青茶  純茶   35   7
2  003     鮮奶綠  鮮奶   55  20
3  004    冬瓜檸檬  果茶   50  14
4  005     多多綠  果茶   50  15
5  006  黑糖珍珠鮮奶  鮮奶   75  27
6  007    芒果冰沙  冰沙   70  29
7  008     熱可可  熱飲   60  21
8  009    檸檬紅茶  果茶   40  11
9  010    芋頭鮮奶  鮮奶   70  26

合併前後筆數： 120927 → 120927 （how=left 時兩者應該相同）


            訂單編號 品項代碼      品項 尺寸  數量  單價  金額
0  T240101059972  004    冬瓜檸檬  中   1  50  50
1  T240101059911  005     多多綠  中   1  50  50
2  T240101059907  006  黑糖珍珠鮮奶  中   1  75  75
3  T240101059907  005     多多綠  大   1  60  60
4  T240101059907  003     鮮奶綠  大   1  65  65


## 任務 3：配不到的代碼——找出主檔沒建檔的門市

**白話說明**：`indicator=True` 會多一欄 `_merge`，告訴你每一筆是「兩邊都有」還是「只有左邊有」。

In [5]:
stores = pd.read_csv('門市主檔.csv')
full = full.merge(stores, on='門市代碼', how='left', indicator=True)
print(full['_merge'].value_counts())

missing = full[full['_merge'] == 'left_only']
print('\n主檔裡找不到的門市代碼：', missing['門市代碼'].unique(),
      '｜筆數：', len(missing), '｜金額：', f"{missing['金額'].sum():,}", '元')
print('最早出現的時間：', missing['訂單時間'].min())

_merge
both          119775
left_only       1152
right_only         0
Name: count, dtype: int64

主檔裡找不到的門市代碼： ['S06'] ｜筆數： 1152 ｜金額： 83,815 元
最早出現的時間： 2024-03-15 10:09:27


> 💡 **看懂結果**：`S06` 是 3/15 開幕的「信義快閃店」，門市主檔還沒建檔。如果用 `how='inner'`，這些營收會**默默消失**在報表裡；用 `how='left'` 加上 `indicator`，才能抓到問題並通知同事補建主檔。
>
> 🤖 **換你問 AI**：「請把訂單明細和門市主檔用門市代碼 merge，以訂單為主，並列出主檔中找不到的門市代碼、筆數與金額。」

In [6]:
full['門市'] = full['門市'].fillna('信義快閃店(S06)')                  # 先補上名稱，避免報表漏掉這筆營收
full = full.drop(columns='_merge')

## 任務 4：pivot_table——做出「門市 × 月份」的季報

In [7]:
report = full.pivot_table(index='門市', columns='月份', values='金額', aggfunc='sum',
                          margins=True, margins_name='合計')[['1月', '2月', '3月', '合計']]   # 加上合計列與合計欄
print((report / 10000).round(1))

growth = (report['3月'] / report['1月'] - 1) * 100
print('\n3 月 vs. 1 月成長率 %：\n', growth.drop('合計').round(1).sort_values(ascending=False))

月份             1月     2月     3月     合計
門市                                    
信義快閃店(S06)    NaN    NaN    8.4    8.4
台中逢甲店        71.5   67.5   78.6  217.5
台北信義店        49.0   46.6   57.1  152.6
台北西門店        55.6   53.6   66.1  175.3
台南成大店        50.0   46.0   55.8  151.7
高雄巨蛋店        56.3   53.0   67.5  176.8
合計          282.3  266.7  333.4  882.4

3 月 vs. 1 月成長率 %：
 門市
高雄巨蛋店         19.9
台北西門店         18.9
台北信義店         16.7
台南成大店         11.7
台中逢甲店          9.9
信義快閃店(S06)     NaN
dtype: float64


## 任務 5：stack / unstack / melt——同一份數字，換三種版面

In [8]:
by_cat = full.groupby(['類別', '月份'])['數量'].sum()                  # 長表：一列 = 一個類別的一個月
print('▶ 長表（groupby 結果）：\n', by_cat.head(6))

wide = by_cat.unstack('月份')[['1月', '2月', '3月']]                    # unstack：把月份搬到欄位 → 寬表（人看的）
wide['成長%'] = ((wide['3月'] / wide['1月'] - 1) * 100).round(1)
print('\n▶ 寬表（類別 × 月份杯數）：\n', wide.sort_values('成長%', ascending=False))

long = wide.drop(columns='成長%').reset_index().melt(id_vars='類別', var_name='月份', value_name='杯數')   # melt：寬表 → 長表（畫圖、存資料庫用）
print('\n▶ melt 回長表：\n', long.head())

with pd.ExcelWriter('第一季季報.xlsx') as w:
    (report / 10000).round(1).to_excel(w, sheet_name='門市x月份(萬元)')
    wide.to_excel(w, sheet_name='類別x月份(杯)')
print('\n✅ 已輸出 第一季季報.xlsx')

▶ 長表（groupby 結果）：
 類別  月份
冰沙  1月      132
    2月      158
    3月     1103
奶茶  1月    12044
    2月    11528
    3月    14452
Name: 數量, dtype: int64

▶ 寬表（類別 × 月份杯數）：
 月份     1月     2月     3月    成長%
類別                            
冰沙    132    158   1103  735.6
純茶   7234   7113   8872   22.6
鮮奶  13199  12498  15950   20.8
奶茶  12044  11528  14452   20.0
果茶  15929  15395  18834   18.2
熱飲   1688    944    212  -87.4

▶ melt 回長表：
    類別  月份     杯數
0  冰沙  1月    132
1  奶茶  1月  12044
2  果茶  1月  15929
3  熱飲  1月   1688
4  純茶  1月   7234

✅ 已輸出 第一季季報.xlsx


> 💡 **看懂結果**：冰沙類從 1 月到 3 月大幅成長（天氣變暖）、熱飲則是衰退——**季節性**非常明顯。進貨與行銷要跟著季節調整。
>
> 🤖 **換你問 AI**：「請把類別 × 月份的杯數做成寬表，新增 3 月比 1 月的成長率欄，並說明 stack、unstack、melt 的差別。」

---

## ✍️ 練習題

1. 用 `concat` 的 `keys` 參數取代手動新增「月份」欄，比較兩種做法的差異。
2. 做一張「門市 × 類別」的毛利樞紐分析表，哪家店的鮮奶系列毛利最高？
3. 把 `report`（不含合計）用 `stack()` 轉成長表，再找出「單月營收最高」的門市與月份。

## 🔑 參考解答（先自己試試看！）

In [9]:
# 練習 1
alt = pd.concat([pd.read_csv(f'訂單_2024-{m}.csv', dtype={'品項代碼': str}) for m in ['01', '02', '03']],
                keys=['1月', '2月', '3月'], names=['月份', '原列號'])
print(alt.head(3))

# 練習 2
profit = full.pivot_table(index='門市', columns='類別', values='毛利', aggfunc='sum')
print(profit.round(0))
print('鮮奶系列毛利最高：', profit['鮮奶'].idxmax())

# 練習 3
s = report.drop(index='合計', columns='合計').stack()
print('單月營收最高：', s.idxmax(), f'{s.max():,.0f} 元')

                 訂單編號                 訂單時間 門市代碼 品項代碼 尺寸  數量
月份 原列號                                                     
1月 0    T240101059972  2024-01-01 10:03:02  S05  004  中   1
   1    T240101059911  2024-01-01 10:06:22  S05  005  中   1
   2    T240101059907  2024-01-01 10:10:32  S05  006  中   1
類別               冰沙        奶茶        果茶       熱飲        純茶        鮮奶
門市                                                                  
信義快閃店(S06)    317.0   14422.0   13931.0     39.0    7960.0   22020.0
台中逢甲店        6960.0  353878.0  460518.0   9980.0  175932.0  541346.0
台北信義店        1413.0  254688.0  261413.0  48552.0  143008.0  362142.0
台北西門店        1361.0  395728.0  327527.0  60893.0  130956.0  324212.0
台南成大店       17864.0  270906.0  363037.0    762.0  126280.0  304780.0
高雄巨蛋店       34698.0  291970.0  431938.0      NaN  149396.0  346428.0
鮮奶系列毛利最高： 台中逢甲店
單月營收最高： ('台中逢甲店', '3月') 785,930 元
